### Precompute text embeddings

We will use both SDXL text encoders embed vector for each caption in the dataset. Then test which is better in training.
The embeds are computed in batches and saved in .pt files.

**UPDATE:**

I came to that both embeds from both encoders have almost same information utility for the model (no significant loss curve difference) so I am using embeds from the first 
encoder as they are slightly smaller and faster to compute.

In [ ]:
import torch
from transformers import CLIPTextModel, CLIPTokenizer, CLIPTextModelWithProjection
import json
import time

In [ ]:
text_encoder = CLIPTextModel.from_pretrained("../../stable-diffusion-xl-base-1.0/text_encoder").to("cuda", torch.float16)
text_encoder_2 = CLIPTextModelWithProjection.from_pretrained("../../stable-diffusion-xl-base-1.0/text_encoder_2").to("cuda", torch.float16)

tokenizer = CLIPTokenizer.from_pretrained("../../stable-diffusion-xl-base-1.0/tokenizer")
tokenizer_2 = CLIPTokenizer.from_pretrained("../../stable-diffusion-xl-base-1.0/tokenizer_2")

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

In [3]:
def process_batch(text_batch):
    inputs = tokenizer(
        text_batch,
        padding="max_length",
        truncation=True,
        max_length=tokenizer.model_max_length,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        embed = text_encoder(inputs.input_ids, output_hidden_states=True, return_dict=True)
        text_embed = embed.pooler_output

    return torch.unbind(text_embed)

In [4]:
text_embeds = process_batch(["a cat on the green grass", "a man"])
print(text_embeds[0].shape)
print(text_embeds[1].shape)

torch.Size([768])
torch.Size([768])


In [ ]:
with open("../../coco2017/annotations/captions_train2017.json") as f:
    data = json.load(f)

In [ ]:
batch_size = 64
embeds = []
batch = []

start = time.time()
for annotation in data["annotations"]:
    caption = annotation["caption"]
    batch.append(caption)
    if len(batch) == batch_size:
        embed_batch = process_batch(batch)
        embeds.extend(embed_batch)
        batch = []

if len(batch) > 0:
    embed_batch = process_batch(batch)
    embeds.extend(embed_batch)
    batch = []

print(time.time() - start)

1.949453353881836


In [ ]:
embeds_array = torch.stack(embeds)
embeds_array = embeds_array.to("cpu", torch.float16)
torch.save(embeds_array, "../../precomputes/embeds_enc1.pt")

In [9]:
embeds_array.shape

torch.Size([21220, 768])

### Compute embeds of second encoder

In [11]:
def process_batch(text_batch):
    inputs_2 = tokenizer_2(
        text_batch,
        padding="max_length",
        truncation=True,
        max_length=tokenizer_2.model_max_length,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        enc2 = text_encoder_2(
            inputs_2.input_ids,
            output_hidden_states=True,
            return_dict=True,
        )

    text_embeds = enc2.text_embeds

    return torch.unbind(text_embeds)

In [12]:
text_embeds = process_batch(["a cat on the green grass", "a man"])
print(text_embeds[0].shape)
print(text_embeds[1].shape)

torch.Size([1280])
torch.Size([1280])


In [ ]:
with open("../../coco2017/annotations/captions_train2017.json") as f:
    data = json.load(f)

In [14]:
batch_size = 64
embeds = []
batch = []

start = time.time()
for annotation in data["annotations"]:
    caption = annotation["caption"]
    batch.append(caption)
    if len(batch) == batch_size:
        embed_batch = process_batch(batch)
        embeds.extend(embed_batch)
        batch = []

if len(batch) > 0:
    embed_batch = process_batch(batch)
    embeds.extend(embed_batch)
    batch = []

print(time.time() - start)

333.95788979530334


In [ ]:
embeds_array = torch.stack(embeds)
embeds_array = embeds_array.to("cpu", torch.float16)
torch.save(embeds_array, "../../precomputes/embeds_enc2.pt")